In [0]:
%pip install requests
dbutils.library.restartPython()

In [0]:
import requests
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_timestamp, lit
from datetime import datetime, timedelta
import time

## Using Databricks Secrets for API Authentication

### Step 1: Create a Secret Scope

You can create a secret scope using the **Databricks CLI** or the **UI**:

#### Option A: Using Databricks CLI
```bash
# Install Databricks CLI if not already installed
pip install databricks-cli

# Configure authentication (one-time setup)
databricks configure --token

# Create a secret scope
databricks secrets create-scope --scope my-api-secrets

# Add your API key to the scope
databricks secrets put --scope my-api-secrets --key api-token
# This will open an editor where you paste your token, then save and close
```

#### Option B: Using the UI (Secrets API)
Navigate to: `https://<your-workspace-url>/#secrets/createScope`

### Step 2: Store Multiple Secrets
```bash
# Store different types of credentials
databricks secrets put --scope my-api-secrets --key api-token
databricks secrets put --scope my-api-secrets --key api-username
databricks secrets put --scope my-api-secrets --key api-password
databricks secrets put --scope my-api-secrets --key base-url
```

### Step 3: List Secrets (to verify)
```bash
# List all scopes
databricks secrets list-scopes

# List secrets in a scope (values are hidden)
databricks secrets list --scope my-api-secrets
```

### Important Notes:
* Secret values are **never displayed** after creation
* Secrets are **encrypted at rest** and **redacted in logs**
* Use **separate scopes** for different environments (dev, staging, prod)
* Grant access to scopes using **ACLs** for team collaboration

In [0]:
# Retrieve secrets from Databricks Secret scope
# NOTE: The actual value is redacted in notebook output for security

# Get API credentials from secrets
api_token = dbutils.secrets.get(scope="my-api-secrets", key="api-token")
api_base_url = dbutils.secrets.get(scope="my-api-secrets", key="base-url")

# For APIs using username/password
api_username = dbutils.secrets.get(scope="my-api-secrets", key="api-username")
api_password = dbutils.secrets.get(scope="my-api-secrets", key="api-password")

print("✓ Secrets loaded successfully (values are redacted in output)")
print(f"API Base URL: {api_base_url}")  # This will show [REDACTED]
print(f"Token length: {len(api_token)} characters")  # This is safe to show

In [0]:
# Example 1: Bearer Token Authentication
def fetch_with_bearer_auth(endpoint):
    api_token = dbutils.secrets.get(scope="my-api-secrets", key="api-token")
    base_url = dbutils.secrets.get(scope="my-api-secrets", key="base-url")
    
    headers = {
        "Authorization": f"Bearer {api_token}",
        "Content-Type": "application/json"
    }
    
    url = f"{base_url}/{endpoint}"
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()

# Example 2: Basic Authentication
def fetch_with_basic_auth(endpoint):
    username = dbutils.secrets.get(scope="my-api-secrets", key="api-username")
    password = dbutils.secrets.get(scope="my-api-secrets", key="api-password")
    base_url = dbutils.secrets.get(scope="my-api-secrets", key="base-url")
    
    from requests.auth import HTTPBasicAuth
    
    url = f"{base_url}/{endpoint}"
    response = requests.get(
        url, 
        auth=HTTPBasicAuth(username, password),
        timeout=30
    )
    response.raise_for_status()
    return response.json()

# Example 3: API Key in Query Parameters
def fetch_with_api_key_param(endpoint):
    api_key = dbutils.secrets.get(scope="my-api-secrets", key="api-token")
    base_url = dbutils.secrets.get(scope="my-api-secrets", key="base-url")
    
    params = {"api_key": api_key}
    url = f"{base_url}/{endpoint}"
    
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

# Example 4: Custom Header Authentication
def fetch_with_custom_header(endpoint):
    api_key = dbutils.secrets.get(scope="my-api-secrets", key="api-token")
    base_url = dbutils.secrets.get(scope="my-api-secrets", key="base-url")
    
    headers = {
        "X-API-Key": api_key,  # Some APIs use custom header names
        "Content-Type": "application/json"
    }
    
    url = f"{base_url}/{endpoint}"
    response = requests.get(url, headers=headers, timeout=30)
    response.raise_for_status()
    return response.json()

print("✓ Authentication functions defined")
print("Choose the appropriate function based on your API's authentication method")

In [0]:
# Example: Fetching data from a public API
def fetch_api_data(api_url, headers=None, params=None):
    """
    Make a GET request to an API endpoint
    
    Args:
        api_url: The API endpoint URL
        headers: Optional dictionary of HTTP headers (e.g., authentication)
        params: Optional dictionary of query parameters
    
    Returns:
        Response data as dictionary
    """
    try:
        response = requests.get(api_url, headers=headers, params=params, timeout=30)
        response.raise_for_status()  # Raise exception for bad status codes
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
        raise

# Example usage with a public API (JSONPlaceholder)
api_url = "https://jsonplaceholder.typicode.com/posts"
data = fetch_api_data(api_url)
print(f"Fetched {len(data)} records")
print(f"Sample record: {data[0]}")

In [0]:
def fetch_paginated_data(base_url, headers=None, max_pages=None):
    """
    Fetch data from a paginated API endpoint
    
    Args:
        base_url: Base API endpoint URL
        headers: Optional authentication/request headers
        max_pages: Maximum number of pages to fetch (None = all)
    
    Returns:
        List of all records across pages
    """
    all_records = []
    page = 1
    
    while True:
        print(f"Fetching page {page}...")
        params = {"_page": page, "_limit": 100}  # Adjust based on your API
        
        response = requests.get(base_url, headers=headers, params=params, timeout=30)
        response.raise_for_status()
        
        data = response.json()
        
        if not data:  # No more data
            break
            
        all_records.extend(data)
        
        # Check if we've reached max pages
        if max_pages and page >= max_pages:
            break
            
        page += 1
        time.sleep(0.5)  # Rate limiting - be respectful to the API
    
    print(f"Total records fetched: {len(all_records)}")
    return all_records

# Example usage
api_url = "https://jsonplaceholder.typicode.com/posts"
all_data = fetch_paginated_data(api_url, max_pages=2)

In [0]:
# Convert JSON data to Spark DataFrame
df = spark.createDataFrame(all_data)

# Add metadata columns
df = df.withColumn("ingestion_timestamp", current_timestamp()) \
       .withColumn("source", lit("api"))

display(df)

In [0]:
# Define your target table location
catalog = "main"  # Update with your catalog
schema = "default"  # Update with your schema
table_name = "api_ingestion_example"

target_table = f"{catalog}.{schema}.{table_name}"

# Write data to Delta table
df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(target_table)

print(f"Data written to {target_table}")

In [0]:
def incremental_api_ingestion(api_url, target_table, id_column="id", headers=None):
    """
    Fetch only new/updated records from API based on last ingestion
    
    Args:
        api_url: API endpoint URL
        target_table: Fully qualified Delta table name
        id_column: Column name used as unique identifier
        headers: Optional API headers
    """
    # Get the maximum ID from existing table (if it exists)
    try:
        max_id = spark.sql(f"SELECT MAX({id_column}) as max_id FROM {target_table}").collect()[0]["max_id"]
        print(f"Last ingested ID: {max_id}")
        
        # Add filter parameter to API request
        params = {f"{id_column}_gte": max_id + 1} if max_id else {}
    except:
        print("Table doesn't exist yet, performing full load")
        params = {}
    
    # Fetch new data
    new_data = fetch_api_data(api_url, headers=headers, params=params)
    
    if not new_data:
        print("No new data to ingest")
        return
    
    # Convert to DataFrame and add metadata
    new_df = spark.createDataFrame(new_data)
    new_df = new_df.withColumn("ingestion_timestamp", current_timestamp())
    
    # Append new records
    new_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(target_table)
    
    print(f"Ingested {new_df.count()} new records")

# Example usage:
# incremental_api_ingestion(
#     api_url="https://api.example.com/data",
#     target_table="main.default.my_api_data",
#     id_column="id",
#     headers={"Authorization": "Bearer YOUR_TOKEN"}
# )

## Best Practices for API Ingestion

### Authentication
* Store API keys/tokens in **Databricks Secrets** - never hardcode credentials
* Use: `dbutils.secrets.get(scope="my_scope", key="api_key")`

### Error Handling
* Implement retry logic with exponential backoff for transient failures
* Log failures to a separate error table for monitoring
* Use try-except blocks to handle API timeouts and connection errors

### Rate Limiting
* Respect API rate limits - add `time.sleep()` between requests
* Consider using libraries like `ratelimit` or `tenacity`

### Incremental Loading
* Use timestamps or IDs to fetch only new/updated records
* Store watermarks (last successful ingestion timestamp) in a control table

### Schema Evolution
* Enable `mergeSchema` option when writing to Delta to handle schema changes
* Consider schema validation before writing to catch breaking changes

### Scheduling
* Use Databricks Jobs to schedule regular API ingestion
* Set appropriate timeout and retry policies

### Data Quality
* Add data validation checks after ingestion
* Track record counts and alert on anomalies